# Nessie & Iceberg Demo - Versionning, Time Travel & Branching

Ce notebook démontre les fonctionnalités clés de **Project Nessie** et **Apache Iceberg**:

1. **Git-like Versioning** - Chaque changement est commité et tracé
2. **Time Travel** - Requêter les données à n'importe quel moment passé
3. **Branching** - Créer des environnements de développement isolés
4. **Merging** - Promouvoir les changements de dev vers production
5. **Rollback** - Revenir à un état précédent

**Prérequis**: Les notebooks 01-04 doivent avoir été exécutés pour créer les tables.

---
## INITIALISATION

In [ ]:
# Configuration du chemin vers le projet
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import des modules Spark et configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET
from pyspark.sql.functions import current_timestamp, current_date, lit, col

# Création de la session Spark
spark = get_spark("nessie-demo")

# Configuration de la branche Nessie main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Création des namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("=" * 60)
print("NESSIE & ICEBERG - DÉMO VERSIONNING")
print("=" * 60)

## PARTIE 1: STRUCTURE DU CATALOGUE NESSIE

Commençons par explorer la structure actuelle du catalogue Nessie.

In [ ]:
# Afficher tous les namespaces dans Nessie
print("=== Namespaces Nessie ===")
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

# Lister toutes les tables
print("=== Tables dans Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("=== Tables dans Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("=== Tables dans Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

## PARTIE 2: BRANCHES NESSIE

### Concept

**Nessie** apporte le versionnage de type Git pour les tables du data lake. Comme Git pour le code, Nessie permet :

* de créer des **branches** isolées pour expérimenter
* de tester des transformations de données sans impacter la production
* de **fusionner (merge)** les modifications validées dans la branche principale

```
┌─────────┐         ┌─────────┐
│  main   │ ←------ │   dev   │
│ (prod)  │  merge  │ (test)  │
└─────────┘         └─────────┘
```

In [ ]:
# Nettoyage de la branche dev si elle existe (pour une démo propre)
spark.sql("DROP BRANCH IF EXISTS dev IN nessie")

print("=== 1. LISTE DES RÉFÉRENCES (BRANCHES) NESSIE ===")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

print("Explication:")
print("- Affiche toutes les branches du catalogue Nessie")
print("- Par défaut, seule 'main' existe (branche de production)")

In [ ]:
# Voir la branche active
spark.sql("USE REFERENCE main IN nessie")

print("=== 2. BRANCHE ACTIVE ===")
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[0]
print(f"Nous sommes sur la branche: {current_ref}")
print("Toutes les opérations SQL affecteront cette branche.")

### Création d'une branche de développement

In [ ]:
print("=== 3. CRÉATION DE LA BRANCHE 'dev' ===")

try:
    spark.sql("CREATE BRANCH dev IN nessie FROM main").show(truncate=False)
    print("Branche 'dev' créée avec succès!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("\nLa branche 'dev' existe déjà (créée précédemment)")
    else:
        print(f"\nInfo: {e}")

print("Explication:")
print("- La branche 'dev' est une copie logique de 'main'")
print("- Pas de copie physique des données (instantané)")
print("- Les modifications sur 'dev' n'impacteront pas 'main'")

### Bascule vers la branche dev

In [ ]:
print("\n=== 4. BASCULE VERS LA BRANCHE 'dev' ===")

spark.sql("USE REFERENCE dev IN nessie")

# Vérification que nous sommes bien sur dev
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

print("Explication:")
print("- Nous sommes maintenant sur la branche 'dev'")
print("- Toutes les prochaines opérations affecteront 'dev'")
print("- La branche 'main' reste intacte (production safe)")

### Modification sur la branche dev

Maintenant que nous sommes sur `dev`, ajoutons des données pour tester notre pipeline. Cette modification n'impactera pas `main`.

In [ ]:
print("=== 5. MODIFICATION SUR LA BRANCHE 'dev' ===")

# Vérification de l'état actuel sur dev
count_dev_before = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' avant: {count_dev_before:,} enregistrements")

# Lecture et ajout d'un nouveau batch sur la branche dev
batch_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch_path)
)

batch_bronze = (
    batch_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_003.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch_bronze.writeTo("nessie.bronze.sales").append()

count_dev_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' après:  {count_dev_after:,} enregistrements")
print(f"Différence: +{count_dev_after - count_dev_before:,} enregistrements")

print("\nCette modification est uniquement sur 'dev' - 'main' n'est pas impactée!")

### Comparaison main vs dev

Vérifions que `main` n'a pas été modifiée en comparant les données des deux branches.

In [ ]:
print("=== 6. COMPARAISON: main vs dev ===")

# Données sur dev (où nous sommes)
dev_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

# Bascule vers main pour comparer
spark.sql("USE REFERENCE main IN nessie")
main_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

print(f"BRANCHE 'dev' (avec Batch 3): {dev_count:,} enregistrements")
print(f"BRANCHE 'main' (production):  {main_count:,} enregistrements")
print(f"\nDifférence: {dev_count - main_count:,} enregistrements de plus sur 'dev'")

# Affichage des batches présents sur main
print("Batches présents sur 'main':")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("La branche 'main' est intacte! Les modifications sur 'dev'")
print("n'ont pas impacté la production.")

### Historique des commits

Affichons l'historique des commits sur la branche active, similaire à `git log`.

In [ ]:
print("=== 7. HISTORIQUE DES COMMITS SUR 'main' ===")
spark.sql("SHOW LOG IN nessie").show(truncate=False)

print("Explication:")
print("- Chaque opération (CREATE, INSERT, MERGE) crée un commit")
print("- Les commits ont un ID, un auteur, un timestamp et un message")
print("- Comme Git, on peut naviguer dans l'historique!")

## PARTIE 3: TIME TRAVEL - VOYAGE DANS LE TEMPS

Avec Iceberg, on peut requêter les données telles qu'elles étaient à n'importe quel moment passé.

In [ ]:
# Affichage de l'historique complet des snapshots Bronze
print("=== HISTORIQUE DES SNAPSHOTS BRONZE ===")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

In [ ]:
# Time Travel: Requêter les données d'un snapshot spécifique
first_snapshot_id = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.history 
    ORDER BY made_current_at 
    LIMIT 1
""").first()[0]

print("\n=== TIME TRAVEL: DONNÉES DU PREMIER SNAPSHOT ===")
count_at_first = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{first_snapshot_id}'
""").first()[0]

print(f"Nombre d'enregistrements au premier snapshot: {count_at_first:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{first_snapshot_id}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

## PARTIE 4: ROLLBACK - RETOUR À UN ÉTAT PRÉCÉDENT

Un des avantages les plus puissants: on peut revenir à un état précédent en cas d'erreur.

In [ ]:
# État actuel avant rollback
print("=== ÉTAT ACTUEL ===")
bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze: {bronze_count:,} enregistrements")

# Récupérer le snapshot précédent
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at
    FROM nessie.bronze.sales.snapshots
    ORDER BY committed_at DESC
""").collect()

if len(snapshots) >= 2:
    previous_snapshot_id = snapshots[1]['snapshot_id']
    
    print(f"\n=== ROLLBACK VERS SNAPSHOT PRÉCÉDENT ===")
    print(f"Snapshot cible: {previous_snapshot_id}")
    
    # Lecture des données à l'état précédent
    bronze_previous = spark.sql(f"""
        SELECT * FROM nessie.bronze.sales VERSION AS OF '{previous_snapshot_id}'
    """)
    
    count_previous = bronze_previous.count()
    print(f"Données à restaurer: {count_previous:,} enregistrements")
    
    # Rollback: remplacement du contenu de la table
    bronze_previous.writeTo("nessie.bronze.sales").using("iceberg").createOrReplace()
    
    print("\nRollback Bronze terminé!")
    
    # Vérification
    count_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
    print(f"Bronze après rollback: {count_after_rollback:,} enregistrements")
    print(f"(Avant rollback: {bronze_count:,})")

## PARTIE 5: FUSION DE BRANCHES

Une fois les tests validés sur `dev`, on peut fusionner la branche dans `main`.

In [ ]:
print("=== 8. FUSION DE 'dev' DANS 'main' ===")

# Fusionner dev dans main
spark.sql("MERGE BRANCH dev INTO main IN nessie").show(truncate=False)

print("Fusion réussie! Les modifications de 'dev' sont maintenant dans 'main'.")

# Vérification que main a maintenant les données
spark.sql("USE REFERENCE main IN nessie")
main_count_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"'main' après fusion: {main_count_after:,} enregistrements")

# Affichage des batches présents sur main après fusion
print("Batches présents sur 'main' après fusion:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Le Batch 3 est maintenant présent en production!")

## RÉSUMÉ - POINTS CLÉS

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║          NESSIE & ICEBERG - FONCTIONNALITÉS DÉMONTRÉES       ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. TIME TRAVEL                                              ║
║     - Requêter les données à un snapshot spécifique          ║
║     - Comparer les snapshots pour voir les changements       ║
║                                                              ║
║  2. BRANCHING (Git-like pour les données)                    ║
║     - Créer des branches pour le développement isolé         ║
║     - Faire des changements sur dev sans impacter main       ║
║     - Fusionner les branches pour promouvoir en production   ║
║                                                              ║
║  3. VERSION CONTROL                                          ║
║     - Chaque écriture crée un nouveau commit                 ║
║     - Historique complet de tous les changements             ║
║     - Retour à n'importe quel état précédent                 ║
║                                                              ║
║  4. ACID TRANSACTIONS                                        ║
║     - Commits atomiques (tout ou rien)                       ║
║     - Lectures cohérentes entre snapshots                    ║
║     - Branches isolées pour le développement concurrent      ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# Retour sur la branche main
spark.sql("USE REFERENCE main IN nessie")
print("Demo terminée. Session Spark connectée à la branche Nessie 'main'.")